# 02 — Covariance Estimator Comparison

Visual and numerical comparison of the three estimators on a single estimation window:
- Sample covariance
- Ledoit-Wolf (shrink toward scaled identity)
- James-Stein (shrink toward single-factor structure)

Inspects shrinkage intensities, eigenvalue spectra, and condition numbers.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.estimation.sample      import sample_covariance
from src.estimation.ledoit_wolf import ledoit_wolf
from src.estimation.james_stein import james_stein

In [ ]:
excess = pd.read_csv("../data/excess_returns_weekly.csv", index_col=0, parse_dates=True)

# Use the most recent 52-week window for inspection
window = excess.iloc[-52:]
print(f"Estimation window: {window.index[0].date()} → {window.index[-1].date()}  ({window.shape[1]} stocks)")

## 1. Compute estimators

In [ ]:
S  = sample_covariance(window)
LW = ledoit_wolf(window)
JS, alpha_js = james_stein(window)

print(f"JS shrinkage intensity α = {alpha_js:.4f}")
print(f"Condition numbers — Sample: {np.linalg.cond(S):.1f}  |  LW: {np.linalg.cond(LW):.1f}  |  JS: {np.linalg.cond(JS):.1f}")

## 2. Eigenvalue spectra

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
matrices = [("Sample", S), ("Ledoit-Wolf", LW), ("James-Stein", JS)]

for ax, (name, M) in zip(axes, matrices):
    eigs = np.sort(np.linalg.eigvalsh(M))[::-1]
    ax.plot(eigs, linewidth=1.2)
    ax.set_title(name)
    ax.set_xlabel("Eigenvalue rank")
    ax.set_yscale("log")

axes[0].set_ylabel("Eigenvalue (log scale)")
plt.suptitle("Eigenvalue Spectra of Covariance Estimators", y=1.02)
plt.tight_layout()
plt.show()

## 3. Covariance heatmaps (top 30 stocks)

In [ ]:
n = 30
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, M) in zip(axes, matrices):
    sns.heatmap(M[:n, :n], ax=ax, cmap="RdBu_r", center=0,
                xticklabels=False, yticklabels=False, cbar=True)
    ax.set_title(name)

plt.suptitle(f"Covariance Heatmaps (top {n} stocks)", y=1.02)
plt.tight_layout()
plt.show()